Experiment-1
Take any large language model (say GPT 3.5) and try to execute some query through it.
Create a small program where you can change the parameter values of Temperature, Top P
and Max Tokens. Please identify how you can make your answer more deterministic?




In [ ]:
!pip install transformers

from transformers import pipeline
import torch

# Initialize the text generation pipeline with a GPT-2 model
text_generator = pipeline('text-generation', model='gpt2')

def generate_ai_text(input_prompt, temp=0.7, nucleus_prob=0.9, max_len=50):
    generated_output = text_generator(
        input_prompt,
        max_length=max_len,
        temperature=temp,
        top_p=nucleus_prob,
        do_sample=True,
        pad_token_id=text_generator.tokenizer.eos_token_id
    )
    return generated_output[0]['generated_text']

from ipywidgets import interact, FloatSlider, IntSlider

@interact
def interactive_generator(
    input_prompt="The future of AI is",
    temp=FloatSlider(min=0.1, max=2.0, step=0.1, value=0.7),
    nucleus_prob=FloatSlider(min=0.1, max=1.0, step=0.1, value=0.9),
    max_len=IntSlider(min=20, max=200, step=10, value=50)
):
    result_text = generate_ai_text(input_prompt, temp, nucleus_prob, max_len)
    print("\nGenerated Text:")
    print(result_text)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

interactive(children=(Text(value='The future of AI is', description='input_prompt'), FloatSlider(value=0.7, de…

Experiment-2
Please identify what are the basic metrics to evaluate your large language model response?
(For example, toxicity, biasness etc). Please write a short program where you can take model
response as input and calculate the score for the above metrics to understand output quality
generated code.

In [ ]:
from transformers import pipeline
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

# Download VADER lexicon
nltk.download('vader_lexicon')

# Initialize sentiment analyzer
sentiment_analyzer = SentimentIntensityAnalyzer()

# Load toxicity and bias classifiers
toxicity_model = pipeline("text-classification", model="unitary/toxic-bert")
bias_model = pipeline("text-classification", model="facebook/roberta-hate-speech-dynabench-r4-target")

def analyze_generated_text(text_response):
    """Evaluates an LLM-generated response for toxicity, bias, and sentiment."""

    # Analyze toxicity
    toxicity_value = toxicity_model(text_response)[0]['score']

    # Analyze bias
    bias_value = bias_model(text_response)[0]['score']

    # Analyze sentiment
    sentiment_value = sentiment_analyzer.polarity_scores(text_response)['compound']

    # Display results
    print("Evaluation Results:")
    print(f"Toxicity: {toxicity_value:.4f} (High = More Toxic)")
    print(f"Bias: {bias_value:.4f} (High = More Biased)")
    print(f"Sentiment: {sentiment_value:.4f} (Positive >0, Negative <0, Neutral ~0)")

# Example Response for Testing
sample_response = "I think some groups of people are just naturally less intelligent."

analyze_generated_text(sample_response)

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


config.json:   0%|          | 0.00/811 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: unitary/toxic-bert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Evaluation Results:
Toxicity: 0.0012 (High = More Toxic)
Bias: 0.9998 (High = More Biased)
Sentiment: 0.4033 (Positive >0, Negative <0, Neutral ~0)


3.Write a program where you can perform keyword-based search. Please take any text file as
input and provide "keyword" dynamically and see whether your algorithm can search it
effectively.

In [ ]:
def find_keyword_in_file(file_name, search_term):
    try:
        with open(file_name, 'r', encoding='utf-8') as file_object:
            is_found = False

            for line_number, line_content in enumerate(file_object, start=1):
                if search_term.lower() in line_content.lower():
                    print(f"Line {line_number}: {line_content.strip()}")
                    is_found = True

            if not is_found:
                print(f"Keyword '{search_term}' not found in the file.")

    except FileNotFoundError:
        print("Error: File not found. Please check the file path.")
    except Exception as error:
        print(f"An error occurred: {error}")

file_name = input("Enter the file path: ")
search_term = input("Enter the keyword: ")

find_keyword_in_file(file_name, search_term)

Enter the file path: /content/machine_learning.txt
Enter the keyword:  machine learning 
Line 6: instructions, machine learning systems identify patterns in data and
Line 34: online stores rely heavily on machine learning algorithms to personalize
Line 37: Despite its advantages, machine learning also faces challenges such as
Line 43: In conclusion, machine learning is a powerful and rapidly evolving field


exp.no:04Write a program where you perform embedding based search. Please take any vector
database and use any embedding technique to search the answer of the query from the given
input text file where query and text files are the inputs of your program

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 35.1 MB/s eta 0:00:00


In [ ]:
import faiss

In [ ]:
from flask import Flask, request, render_template
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Initialize Flask app
web_app = Flask(__name__)

# Load embedding model
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

# Function to read the text file
def load_text_lines(file_name):
    with open(file_name, 'r', encoding='utf-8') as file_obj:
        return file_obj.readlines()

# Function to generate embeddings
def create_embeddings(text_list):
    return sentence_model.encode(text_list, convert_to_numpy=True)

# Function to build FAISS index
def create_faiss_index(embedding_array):
    dimension = embedding_array.shape[1]
    faiss_index = faiss.IndexFlatL2(dimension)
    faiss_index.add(embedding_array)
    return faiss_index

# 🔹 Load and process text data (CORRECT PLACE)
input_file = "/content/machine_learning.txt"   # or "/content/machine_learning.txt" for Colab
text_list = load_text_lines(input_file)
embedding_array = create_embeddings(text_list)
faiss_index = create_faiss_index(embedding_array)

# Function to perform search
def perform_search(user_query, text_list, faiss_index, top_k=5):
    query_vector = sentence_model.encode([user_query], convert_to_numpy=True)
    distances, indices = faiss_index.search(query_vector, top_k)
    return [(text_list[i], float(distances[0][j])) for j, i in enumerate(indices[0])]

# Flask route
@web_app.route("/", methods=["GET", "POST"])
def home_page():
    search_results = []
    user_query = ""

    if request.method == "POST":
        user_query = request.form["query"]
        search_results = perform_search(user_query, text_list, faiss_index)

    return render_template("index.html", query=user_query, results=search_results)

# Run app
if __name__ == "__main__":
    web_app.run(debug=True, port=5005)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5005
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)


 exp no:5
 Please take 2/3 medical reports (may be blood reports) and store them in a place. Please write
a program which can read all the files dynamically from the given locations. Please try to
understand the metadata of the reports

In [ ]:
import os
import csv

# 🔹 Create folder in Colab
folder_path = "medical_reports"
os.makedirs(folder_path, exist_ok=True)

# 🔹 Create sample CSV file
file_path = os.path.join(folder_path, "report1.csv")

with open(file_path, "w") as f:
    f.write("Patient Name,John Doe\n")
    f.write("Report Date,2026-04-08\n\n")
    f.write("Test,Result,Unit,Reference Range\n")
    f.write("Hemoglobin,13.5,g/dL,13-17\n")
    f.write("WBC,6000,cells/mcL,4000-11000\n")
    f.write("Platelets,250000,cells/mcL,150000-450000\n")


# 🔹 Function to read reports
def load_medical_data(folder_path):
    report_list = []

    for file_name in os.listdir(folder_path):
        if file_name.endswith('.csv'):
            file_path = os.path.join(folder_path, file_name)

            with open(file_path, 'r') as file_handle:
                csv_reader = csv.reader(file_handle)
                file_lines = list(csv_reader)

                patient = file_lines[0][1].strip()
                date = file_lines[1][1].strip()
                test_rows = file_lines[3:]

                test_list = [
                    {
                        "Test_Name": row[0].strip(),
                        "Test_Result": row[1].strip(),
                        "Unit": row[2].strip(),
                        "Reference": row[3].strip()
                    }
                    for row in test_rows if len(row) >= 4
                ]

                report_data = {
                    "File_Name": file_name,
                    "Patient": patient,
                    "Date": date,
                    "Test_Details": test_list
                }

                report_list.append(report_data)

    return report_list


# 🔹 Run program
reports = load_medical_data(folder_path)

for report in reports:
    print(f"\n--- Report: {report['File_Name']} ---")
    print(f"Patient: {report['Patient']}")
    print(f"Date: {report['Date']}")
    print("Test Results:")

    for test in report['Test_Details']:
        print(f"  - {test['Test_Name']}: {test['Test_Result']} {test['Unit']} (Ref: {test['Reference']})")


--- Report: report1.csv ---
Patient: John Doe
Date: 2026-04-08
Test Results:
  - Test: Result Unit (Ref: Reference Range)
  - Hemoglobin: 13.5 g/dL (Ref: 13-17)
  - WBC: 6000 cells/mcL (Ref: 4000-11000)
  - Platelets: 250000 cells/mcL (Ref: 150000-450000)


4th exp


In [ ]:
!pip install -q sentence-transformers faiss-cpu numpy

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import os

# Load model
encoder_model = SentenceTransformer("all-MiniLM-L6-v2")

# Upload file (for Google Colab users)
try:
    from google.colab import files
    print("Upload your text file:")
    files.upload()
except:
    pass  # Skip if not in Colab

# Function to read file safely
def read_text_file(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            lines = [line.strip() for line in file.readlines() if line.strip()]
        return lines
    except FileNotFoundError:
        print("❌ Error: File not found. Please check the path.")
        print("📂 Current directory:", os.getcwd())
        return []

# Generate embeddings
def generate_embeddings(texts):
    embeddings = encoder_model.encode(texts, convert_to_numpy=True)
    return embeddings.astype("float32")

# Build FAISS index
def build_faiss_index(embeddings):
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    return index

# Search function
def search(query, texts, index, top_k=3):
    query_embedding = encoder_model.encode(
        [query], convert_to_numpy=True
    ).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append((texts[idx], float(distances[0][rank])))
    return results

# User input
text_file_path = input("Enter the path of the text file: ")
query = input("Enter your search query: ")

# Read file
texts = read_text_file(text_file_path)

# Stop if file not loaded
if not texts:
    print("⚠️ No data to search. Please fix the file path.")
else:
    # Process
    embeddings = generate_embeddings(texts)
    index = build_faiss_index(embeddings)

    results = search(query, texts, index, top_k=3)

    # Output
    print("\n🔍 Top Matching Results:\n")
    for i, (text, score) in enumerate(results, start=1):
        print(f"{i}. {text}")
        print(f"   Distance Score: {score:.4f}\n")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Upload your text file:


Saving Machine learning is a method of dat.txt to Machine learning is a method of dat.txt
Enter the path of the text file: /content/Machine learning is a method of dat.txt
Enter your search query: machine learning

🔍 Top Matching Results:

1. Machine learning is a method of data analysis that automates analytical model building.
   Distance Score: 0.7747

2. like statistical analysis, predictive modeling, and machine learning.
   Distance Score: 1.0770

3. Reinforcement learning is an area of machine learning where an agent learns by interacting
   Distance Score: 1.1237

